<!-- KERNEL_BANNER -->
> **Use kernel: `mrigi_tor190_v8`**
>
> Set the notebook kernel to *Python (mrigi_tor190_v8)* before running.

# 23M: Llama-3-8B-Instruct replication run — no-context vs MMR, correctness only

## Why

In the reported results, stock `Llama-3-8B-Instruct` scores **7.41 in both conditions**
under `gpt-4o` — a mean difference of exactly 0.00. The underlying data is not degenerate
(81 of 100 per-question scores change; 45 up, 36 down; Wilcoxon p = 0.96), so the null
result is genuine, but a single run cannot distinguish *"retrieval truly has no net effect"*
from *"the effect is smaller than run-to-run noise."*

This notebook re-runs the same model on the same 100 questions and re-judges it, to
measure that noise floor directly.

## What it does

1. **Regenerates** `Llama-3-8B-Instruct` answers for both conditions, using the identical
   question set, prompts, retriever and decoding parameters as 23j.
2. **Extracts** answers with the *fixed* prompt-anchored extractor from 30g — so this run
   never carries the echo bug.
3. **Judges correctness** with `gpt-4o` and the verbatim 30g prompt.
4. **Compares** replicate vs original: per-question agreement, the no-context − MMR gap in
   each run, and whether the null result reproduces.

## Read this before interpreting the output

Generation uses `temperature=0.1, do_sample=True` — **sampling is on**, so the replicate
will not be identical to the original. That is the point: the run-to-run spread *is* the
measurement. Set `DETERMINISTIC = True` below to use greedy decoding instead, which removes
sampling noise but no longer matches the protocol used for the reported results.

**Cost:** 200 generations (~10 min GPU) + 200 `gpt-4o` judge calls.

In [1]:
import os
# torch 1.12 in this kernel: do NOT set PYTORCH_CUDA_ALLOC_CONF=expandable_segments.
import torch, json, glob, gc, time, httpx
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from scipy.stats import wilcoxon, pearsonr

from langchain.llms import HuggingFacePipeline
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

with open('/home/jupyter/Mrigi/env.sh') as f:
    for line in f:
        line = line.strip()
        if line.startswith('export ') and '=' in line:
            k, v = line[len('export '):].split('=', 1)
            os.environ[k] = v.strip('"').strip("'")
hf_token = os.environ['HF_TOKEN_BESTE']
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
OPENAI_KEY = os.environ['OPENAI_API_TOKEN']
HEADERS = {'Authorization': f'Bearer {OPENAI_KEY}', 'Content-Type': 'application/json'}
API = 'https://api.openai.com/v1/chat/completions'

MODEL_REPO    = 'meta-llama/Meta-Llama-3-8B-Instruct'
MODEL_LABEL   = 'Llama-3-8B-Instruct'
JUDGE         = 'gpt-4o'
K_MMR         = 15
MAX_NEW       = 400
DETERMINISTIC = False    # True -> greedy decoding (no sampling noise, but off-protocol)
WORKERS       = 8
TS            = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}')
print(f'decoding: {"greedy (do_sample=False)" if DETERMINISTIC else "sampling, temperature=0.1 (matches 23j)"}')

probe = httpx.post(API, headers=HEADERS, timeout=60,
                   json={'model': 'gpt-4o-mini', 'messages': [{'role': 'user', 'content': 'hi'}],
                         'max_tokens': 1})
if probe.status_code != 200:
    raise RuntimeError(f'API unusable (HTTP {probe.status_code}): '
                       f'{probe.json().get("error", {}).get("message", "")[:160]}')
print('\u2713 billing live')

torch 1.12.1  cuda=True
decoding: sampling, temperature=0.1 (matches 23j)
✓ billing live


In [2]:
# ---------------------------------------------------------------------------
# Questions and gold answers are taken from the CLEAN results file, not re-read
# from the spreadsheet. That guarantees this run is scored on byte-identical
# inputs to the original - including the post-23L Q67 - so any difference in the
# result is attributable to generation, not to the question set.
# ---------------------------------------------------------------------------
CLEAN_FILE = sorted(glob.glob('results_23j_clean_*.json'), key=os.path.getmtime)[-1]
orig = json.load(open(CLEAN_FILE))['results'][MODEL_LABEL]

QUESTIONS = [{'q_idx': i, 'query': r['query'], 'gold': r['gold'],
              'title': r.get('title'), 'doi': r.get('doi')}
             for i, r in enumerate(orig['no_context'])]

print(f'question source : {CLEAN_FILE}')
print(f'questions       : {len(QUESTIONS)}')
print(f'Q67             : {QUESTIONS[67]["query"][:90]}...')

embeddings = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')
vdb = FAISS.load_local(Path('faiss_index'), embeddings, index_name='index',
                       allow_dangerous_deserialization=True)
print(f'FAISS index     : {vdb.index.ntotal:,} vectors')


def mmr_context(q):
    return '\n\n'.join(d.page_content for d in vdb.max_marginal_relevance_search(q, k=K_MMR))


# Retrieval is deterministic given the query, so contexts are built once and reused.
print('building MMR contexts...')
CONTEXTS = [mmr_context(q['query']) for q in tqdm(QUESTIONS, desc='MMR')]
print(f'mean context length: {np.mean([len(c) for c in CONTEXTS]):,.0f} chars')

question source : results_23j_clean_20260803_194431.json
questions       : 100
Q67             : In Si-rich SSZ-13, why can some framework Al arrangements fail to generate exchange sites ...


/tmp/ipykernel_1340486/371508125.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name='all-MiniLM-L6-v2')


FAISS index     : 1,474,439 vectors
building MMR contexts...


MMR: 100%|██████████| 100/100 [00:15<00:00,  6.26it/s]

mean context length: 13,922 chars


In [3]:
# Prompts verbatim from 23j; extractor verbatim from 30g (the fixed one).
WITH_CTX = """You are an expert on zeolite synthesis, chemistry, and catalysis. Answer the following question using the provided context together with your own knowledge. Give a clear, focused answer in 3\u20135 sentences (or fewer if the question is simple). Do not invent citations. Do not restate the question.

Context:
{context}

Question: {question}

Answer:"""

NO_CTX = """You are an expert on zeolite synthesis, chemistry, and catalysis. Answer the following question using your own knowledge. Give a clear, focused answer in 3\u20135 sentences (or fewer if the question is simple). Do not invent citations. Do not restate the question.

Question: {question}

Answer:"""


def extract_answer(text):
    """Prompt-anchored extraction (30g). Both prompts end in 'Answer:'; instruct models
    then emit a bare 'assistant' marker because the pipeline decodes the special tokens
    away. Splitting on the LAST 'Answer:' handles it; the context block never ends there."""
    if not isinstance(text, str):
        return None
    i = text.rfind('Answer:')
    t = (text[i + len('Answer:'):] if i != -1 else text).lstrip()
    if t.startswith('assistant'):
        t = t[len('assistant'):].lstrip()
    for tok in ('<|eot_id|>', '<|end_of_text|>', '<|begin_of_text|>',
                '<|start_header_id|>', '<|end_header_id|>'):
        t = t.replace(tok, '')
    return t.strip()


print('\u2713 prompts + fixed extractor loaded')

✓ prompts + fixed extractor loaded


## Step 1 — Regenerate

In [ ]:
cached = sorted(glob.glob('results_23M_llama_replicate_*.json'), key=os.path.getmtime)
replicate = None
if cached:
    p = json.load(open(cached[-1]))
    if p.get('n_questions') == len(QUESTIONS):
        replicate = p['results']
        print(f'reusing cached generation: {cached[-1]}')

if replicate is None:
    print(f'loading {MODEL_REPO} ...')
    tok = AutoTokenizer.from_pretrained(MODEL_REPO, use_fast=True, trust_remote_code=True,
                                        token=hf_token, local_files_only=True)
    mdl = AutoModelForCausalLM.from_pretrained(MODEL_REPO, device_map='cuda:0',
                                               torch_dtype=torch.float16, trust_remote_code=True,
                                               token=hf_token, local_files_only=True)
    mdl.eval()
    gen_kwargs = dict(max_new_tokens=MAX_NEW)
    if DETERMINISTIC:
        gen_kwargs['do_sample'] = False
    else:
        gen_kwargs.update(do_sample=True, temperature=0.1)
    pipe = pipeline('text-generation', model=mdl, tokenizer=tok, **gen_kwargs)
    llm = HuggingFacePipeline(pipeline=pipe)
    print(f'\u2713 loaded  ({torch.cuda.memory_allocated(0)/1e9:.1f} GB)')

    replicate = {'no_context': [], 'mmr': []}
    for cond in ('no_context', 'mmr'):
        for j, q in enumerate(tqdm(QUESTIONS, desc=f'generating {cond}')):
            prompt = (NO_CTX.format(question=q['query']) if cond == 'no_context'
                      else WITH_CTX.format(context=CONTEXTS[j], question=q['query']))
            formatted = tok.apply_chat_template([{'role': 'user', 'content': prompt}],
                                                tokenize=False, add_generation_prompt=True)
            try:
                ans = extract_answer(llm(formatted))
                ans = ans if ans else 'INVALID'
            except Exception as e:
                print(f'  q{j} error: {str(e)[:90]}')
                ans = 'ERROR'
            replicate[cond].append({**q, 'answer': ans, 'answer_len': len(ans)})
            gc.collect(); torch.cuda.empty_cache()

    try:
        mdl.to('cpu'); pipe.model = None; pipe.tokenizer = None
    except Exception:
        pass
    del llm, pipe, mdl, tok
    for _ in range(3):
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()

    out = f'results_23M_llama_replicate_{TS}.json'
    json.dump({'model': MODEL_LABEL, 'repo': MODEL_REPO, 'timestamp': TS,
               'deterministic': DETERMINISTIC, 'k_mmr': K_MMR, 'max_new_tokens': MAX_NEW,
               'question_source': CLEAN_FILE, 'n_questions': len(QUESTIONS),
               'results': replicate}, open(out, 'w'), indent=2)
    print(f'\u2713 saved {out}')

for cond in ('no_context', 'mmr'):
    ok = [r for r in replicate[cond] if r['answer'] not in ('ERROR', 'INVALID')]
    print(f'  {cond:<11} valid {len(ok)}/{len(replicate[cond])}   mean {np.mean([r["answer_len"] for r in ok]):.0f} chars')

loading meta-llama/Meta-Llama-3-8B-Instruct ...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v8/lib/python3.9/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
/tmp/ipykernel_1340486/3771678362.py:23: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


✓ loaded  (16.3 GB)


generating no_context:   0%|          | 0/100 [00:00<?, ?it/s]Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
/tmp/ipykernel_1340486/3771678362.py:34: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  ans = extract_answer(llm(formatted))
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
generating no_context:   1%|          | 1/100 [00:03<06:34,  3.98s/it]Using unk_token, but it is not set yet.
Using sep_token, but it is not set yet.
Using pad_token, but it is not set yet.
Using cls_token, but it is not set yet.
Using mask_token, but it is not set yet.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
generating no_context:   2%|▏         | 2/100 [00:08<06:52,  4.20s/it]Using unk_token

## Step 2 — Judge correctness (`gpt-4o`, verbatim 30g prompt)

In [ ]:
JUDGE_SYSTEM_PROMPT = """You are an expert evaluator for a Retrieval-Augmented Generation (RAG) system 
focused on zeolite synthesis, catalysis, and environmental applications. 
You will evaluate the quality of retrieved contexts and open-ended generated responses 
against a reference (gold) answer.
Always respond in valid JSON format."""

CORRECTNESS = """Given the following query, a reference (gold) answer, and a model's open-ended response, rate how correct the response is.

Judge on scientific substance, not wording. The response may phrase things differently, be more or less detailed, or add related information \u2014 that is fine as long as the core claim matches the reference and any additional claims are accurate.

**Scoring Rubric (1-10):**
- 1-2: Completely incorrect \u2014 asserts something that contradicts the reference answer
- 3-4: Mostly incorrect \u2014 misses the main point, but shows partial understanding of the topic
- 5-6: Partially correct \u2014 captures part of the reference answer but is missing or muddling the key idea
- 7-8: Mostly correct \u2014 captures the key idea of the reference answer, with minor gaps or imprecise phrasing
- 9-10: Fully correct \u2014 captures the reference answer's key idea clearly and accurately (extra correct detail is fine)

**Query:**
{query}

**Reference (Gold) Answer:**
{gold_answer}

**Model's Open-Ended Response:**
{response}

Respond with ONLY a JSON object in this exact format:
{{"score": <integer 1-10>, "reasoning": "<one to two sentences explaining your score>"}}"""


def call_judge(user, retries=4):
    body = {'model': JUDGE,
            'messages': [{'role': 'system', 'content': JUDGE_SYSTEM_PROMPT},
                         {'role': 'user', 'content': user}],
            'temperature': 0.1, 'max_tokens': 200,
            'response_format': {'type': 'json_object'}}
    for a in range(retries):
        try:
            r = httpx.post(API, headers=HEADERS, json=body, timeout=120)
        except Exception:
            time.sleep(3 * (a + 1)); continue
        if r.status_code == 200:
            try:
                d = json.loads(r.json()['choices'][0]['message']['content'].strip())
                s = int(d.get('score', -1))
                return (s if 1 <= s <= 10 else -1), d.get('reasoning', '')
            except Exception as e:
                return -1, f'parse: {e}'
        msg = r.json().get('error', {}).get('message', '')
        if r.status_code == 429 and 'credit' in msg.lower():
            return -1, 'billing'
        time.sleep(4 * (a + 1))
    return -1, 'retries exceeded'


def judge_one(key):
    cond, i = key
    rec = replicate[cond][i]
    if rec['answer'] in ('ERROR', 'INVALID'):
        return key, -1
    s, _ = call_judge(CORRECTNESS.format(query=rec['query'], gold_answer=rec['gold'],
                                         response=rec['answer']))
    return key, s


plan = [(c, i) for c in ('no_context', 'mmr') for i in range(len(QUESTIONS))]
rep_scores = {}
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = [ex.submit(judge_one, k) for k in plan]
    for fut in tqdm(as_completed(futs), total=len(futs), desc=f'judging ({JUDGE})'):
        k, s = fut.result(); rep_scores[k] = s

json.dump({'model': MODEL_LABEL, 'judge': JUDGE, 'task': 'correctness', 'timestamp': TS,
           'deterministic': DETERMINISTIC,
           'results': {f'{c}|{i}': v for (c, i), v in rep_scores.items()}},
          open(f'judge_results_23M_{TS}.json', 'w'), indent=2)
print(f'\u2713 saved judge_results_23M_{TS}.json')
for c in ('no_context', 'mmr'):
    v = [rep_scores[(c, i)] for i in range(len(QUESTIONS)) if rep_scores[(c, i)] > 0]
    print(f'  {c:<11} mean {np.mean(v):.2f}   n={len(v)}')

## Step 3 — Replicate vs original: is the null result stable?

In [ ]:
ORIG_JUDGE = sorted(glob.glob('judge_results_23j_clean_*.json'), key=os.path.getmtime)[-1]
oj = json.load(open(ORIG_JUDGE))['results']


def orig_scores(cond):
    return {int(k.split('|')[2]): oj[k][JUDGE]['correctness']['score']
            for k in oj if k.startswith(f'{MODEL_LABEL}|{cond}|')
            and oj[k][JUDGE]['correctness']['score'] > 0}


def rep(cond):
    return {i: rep_scores[(cond, i)] for i in range(len(QUESTIONS)) if rep_scores[(cond, i)] > 0}


print(f'original judge file : {ORIG_JUDGE}')
print(f'judge               : {JUDGE}\n')

print(f'{"":<14}{"original":>10}{"replicate":>11}{"diff":>8}')
print('-' * 43)
means = {}
for cond in ('no_context', 'mmr'):
    o, rp = orig_scores(cond), rep(cond)
    means[cond] = (np.mean(list(o.values())), np.mean(list(rp.values())))
    print(f'{cond:<14}{means[cond][0]:>10.2f}{means[cond][1]:>11.2f}{means[cond][1]-means[cond][0]:>+8.2f}')

gap_o = means['mmr'][0] - means['no_context'][0]
gap_r = means['mmr'][1] - means['no_context'][1]
print(f'\nMMR \u2212 no-context gap:   original {gap_o:+.2f}   replicate {gap_r:+.2f}')

# Does the null reproduce within the replicate?
o_nc, o_m = rep('no_context'), rep('mmr')
common = sorted(set(o_nc) & set(o_m))
d = np.array([o_m[i] - o_nc[i] for i in common], float)
p = wilcoxon(d).pvalue if np.any(d) else 1.0
print(f'\nWithin the REPLICATE: MMR \u2212 no-context = {d.mean():+.2f}  '
      f'(up {int((d>0).sum())}, down {int((d<0).sum())}, tie {int((d==0).sum())})  p={p:.3f}')
print(f'  -> {"NULL REPRODUCES (no significant effect)" if p >= 0.05 else "SIGNIFICANT in the replicate - the original null may be underpowered"}')

# Run-to-run noise floor
print('\n=== run-to-run stability (same model, same questions, resampled) ===')
for cond in ('no_context', 'mmr'):
    o, rp = orig_scores(cond), rep(cond)
    c = sorted(set(o) & set(rp))
    a = np.array([o[i] for i in c], float); b = np.array([rp[i] for i in c], float)
    print(f'  {cond:<11} same score on {int((a==b).sum())}/{len(c)} questions  '
          f'|mean diff| {abs(b.mean()-a.mean()):.2f}  r={pearsonr(a,b)[0]:.3f}  '
          f'mean |per-q diff| {np.mean(np.abs(b-a)):.2f}')

noise = abs(gap_r - gap_o)
print(f'\nThe condition gap moved {noise:.2f} points between runs.')
print(f'Any reported effect smaller than ~{noise:.2f} points is within single-run noise')
print('for this pipeline and should not be interpreted.')

In [ ]:
print('=' * 74)
print('23M complete')
print('=' * 74)
print(f'model      : {MODEL_LABEL}')
print(f'decoding   : {"greedy" if DETERMINISTIC else "sampling, temperature=0.1 (matches 23j)"}')
print(f'judge      : {JUDGE}, correctness only')
print(f'questions  : {len(QUESTIONS)} (from {CLEAN_FILE})')
print(f'\ngenerations : results_23M_llama_replicate_{TS}.json')
print(f'judgments   : judge_results_23M_{TS}.json')
print('\nNote: this run is NOT merged into results_23j_checkpoint.json or the clean')
print('judge file - it is a standalone replicate for estimating run-to-run variance.')
print('=' * 74)